In [5]:
import os
print(os.getcwd())

/Users/sandeepa/travel-weather-planner/notebook


In [6]:
import os
import pandas as pd

# Each pair: a famous destination and a less-crowded alternative
destinations = pd.DataFrame([
    {"pair_id": 1, "name": "Goa",            "country": "India",     "type": "famous", "lat": 15.49, "lon": 73.83},
    {"pair_id": 1, "name": "Gokarna",        "country": "India",     "type": "hidden", "lat": 14.55, "lon": 74.32},
    {"pair_id": 2, "name": "Manali",         "country": "India",     "type": "famous", "lat": 32.24, "lon": 77.19},
    {"pair_id": 2, "name": "Tirthan Valley", "country": "India",     "type": "hidden", "lat": 31.63, "lon": 77.45},
    {"pair_id": 3, "name": "Jaipur",         "country": "India",     "type": "famous", "lat": 26.91, "lon": 75.79},
    {"pair_id": 3, "name": "Bundi",          "country": "India",     "type": "hidden", "lat": 25.43, "lon": 75.65},
    {"pair_id": 4, "name": "Bali",           "country": "Indonesia", "type": "famous", "lat": -8.65, "lon": 115.22},
    {"pair_id": 4, "name": "Lombok",         "country": "Indonesia", "type": "hidden", "lat": -8.58, "lon": 116.12},
    {"pair_id": 5, "name": "Phuket",         "country": "Thailand",  "type": "famous", "lat": 7.88,  "lon": 98.39},
    {"pair_id": 5, "name": "Koh Lanta",      "country": "Thailand",  "type": "hidden", "lat": 7.62,  "lon": 99.05},
    {"pair_id": 6, "name": "Santorini",      "country": "Greece",    "type": "famous", "lat": 36.42, "lon": 25.43},
    {"pair_id": 6, "name": "Milos",          "country": "Greece",    "type": "hidden", "lat": 36.73, "lon": 24.45},
    {"pair_id": 7, "name": "Dubai",          "country": "UAE",       "type": "famous", "lat": 25.20, "lon": 55.27},
    {"pair_id": 7, "name": "Muscat",         "country": "Oman",      "type": "hidden", "lat": 23.59, "lon": 58.38},
    {"pair_id": 8, "name": "Kyoto",          "country": "Japan",     "type": "famous", "lat": 35.01, "lon": 135.77},
    {"pair_id": 8, "name": "Kanazawa",       "country": "Japan",     "type": "hidden", "lat": 36.56, "lon": 136.66},
])

os.makedirs("../data/raw", exist_ok=True)   # create the folder if it does not exist
destinations.to_csv("../data/raw/destinations.csv", index=False)
destinations

,pair_id,name,country,type,lat,lon
0,1,Goa,India,famous,15.49,73.83
1,1,Gokarna,India,hidden,14.55,74.32
2,2,Manali,India,famous,32.24,77.19
3,2,Tirthan Valley,India,hidden,31.63,77.45
4,3,Jaipur,India,famous,26.91,75.79
5,3,Bundi,India,hidden,25.43,75.65
6,4,Bali,Indonesia,famous,-8.65,115.22
7,4,Lombok,Indonesia,hidden,-8.58,116.12
8,5,Phuket,Thailand,famous,7.88,98.39
9,5,Koh Lanta,Thailand,hidden,7.62,99.05


In [7]:
import os
os.makedirs("../data/raw", exist_ok=True)   # create the folder if it does not exist

destinations.to_csv("../data/raw/destinations.csv", index=False)

In [8]:
import requests
import time

url = "https://archive-api.open-meteo.com/v1/archive"
all_weather = []   # we will collect one table per destination here

for _, row in destinations.iterrows():
    params = {
        "latitude": row["lat"],
        "longitude": row["lon"],
        "start_date": "2021-01-01",
        "end_date": "2025-12-31",
        "daily": ("temperature_2m_max,temperature_2m_min,"
                  "precipitation_sum,sunshine_duration"),
        "timezone": "auto",   # use each destination's local time
    }
    response = requests.get(url, params=params)

    if response.status_code != 200:
        print("Failed:", row["name"], response.status_code)
        continue

    city = pd.DataFrame(response.json()["daily"])
    city["name"] = row["name"]    # remember which place this data belongs to
    all_weather.append(city)
    print("Downloaded:", row["name"], len(city), "days")

    time.sleep(1)   # short pause, to be polite to the free API

weather = pd.concat(all_weather, ignore_index=True)
weather = weather.rename(columns={"time": "date"})
weather["date"] = pd.to_datetime(weather["date"])

weather.to_csv("../data/raw/weather_raw.csv", index=False)
print(weather.shape)
weather.head()

Downloaded: Goa 1826 days
Downloaded: Gokarna 1826 days
Downloaded: Manali 1826 days
Downloaded: Tirthan Valley 1826 days
Downloaded: Jaipur 1826 days
Downloaded: Bundi 1826 days
Downloaded: Bali 1826 days
Downloaded: Lombok 1826 days
Downloaded: Phuket 1826 days
Downloaded: Koh Lanta 1826 days
Downloaded: Santorini 1826 days
Downloaded: Milos 1826 days
Downloaded: Dubai 1826 days
Downloaded: Muscat 1826 days
Downloaded: Kyoto 1826 days
Downloaded: Kanazawa 1826 days
(29216, 6)


,date,temperature_2m_max,temperature_2m_min,precipitation_sum,sunshine_duration,name
0,2021-01-01,32.0,24.8,0.0,37910.33,Goa
1,2021-01-02,31.5,22.3,0.0,38566.88,Goa
2,2021-01-03,31.3,22.2,0.0,33505.28,Goa
3,2021-01-04,28.4,22.7,3.5,9360.56,Goa
4,2021-01-05,30.0,24.0,5.0,36783.40,Goa


In [9]:
weather = weather.rename(columns={
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "precipitation_sum": "rain_mm",
})


weather["sunshine_hours"] = (weather["sunshine_duration"]/3600).round(1)
weather = weather.drop(columns=["sunshine_duration"])

weather["year"] = weather["date"].dt.year
weather["month"] = weather["date"].dt.month

print(weather.isna().sum())
weather.head()

date              0
temp_max          0
temp_min          0
rain_mm           0
name              0
sunshine_hours    0
year              0
month             0
dtype: int64


,date,temp_max,temp_min,rain_mm,name,sunshine_hours,year,month
0,2021-01-01,32.0,24.8,0.0,Goa,10.5,2021,1
1,2021-01-02,31.5,22.3,0.0,Goa,10.7,2021,1
2,2021-01-03,31.3,22.2,0.0,Goa,9.3,2021,1
3,2021-01-04,28.4,22.7,3.5,Goa,2.6,2021,1
4,2021-01-05,30.0,24.0,5.0,Goa,10.2,2021,1


In [10]:
os.makedirs("../data/processed", exist_ok=True)

# Save columns in a fixed order (this order will match our SQL table)
columns = ["name", "date", "year", "month", "temp_max", "temp_min", "rain_mm", "sunshine_hours"]
weather[columns].to_csv("../data/processed/weather_clean.csv", index=False)
print("Saved!")

Saved!


In [11]:
from getpass import getpass
from sqlalchemy import create_engine, URL

# Ask for the password instead of writing it in the code
password = getpass("PostgreSQL password: ")

db_url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="travel_planner",
)
engine = create_engine(db_url)

# Send both tables to PostgreSQL
destinations.to_sql("destinations", engine, if_exists="append", index=False)
weather[columns].to_sql("weather", engine, if_exists="append", index=False, chunksize=5000)
print("Uploaded!")

PostgreSQL password:  ········


Uploaded!
